# Treino no Google Colab: triagem de retinopatia diabética

> ⚠️ Projeto educacional. Não é dispositivo médico.

Treina a EfficientNet-B0 com o APTOS 2019 e baixa `models/best.pt` e os resultados.

**Antes de começar:**
1. Em *Ambiente de execução > Alterar o tipo de ambiente de execução*, escolha **GPU**.
2. Aceite as regras em <https://www.kaggle.com/competitions/aptos2019-blindness-detection/rules> (sem isso, o download dá erro 403).
3. Gere um token de API em <https://www.kaggle.com/settings>.
4. Confira `REPO_URL` na célula 2. Se o repositório for privado, torne-o público ou use uma URL com token.

O download tem cerca de 10 GB. Os arquivos do Colab somem quando a sessão termina, então baixe os resultados no final ou use a célula opcional do Google Drive.

## 1. Confere a GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "Sem GPU: troque o tipo de ambiente de execução para GPU."
print(torch.__version__, torch.cuda.get_device_name(0))

## 2. Baixa o código

O Colab já traz torch, OpenCV, pandas, scikit-learn e matplotlib. Não instale `requirements.txt` aqui: ele trocaria o PyTorch com CUDA do Colab.

In [ ]:
REPO_URL = "https://github.com/richprograming/dr-screening.git"

!git clone --depth 1 {REPO_URL} /content/dr-screening
%cd /content/dr-screening

## 3. Baixa o APTOS 2019

Cole o token quando for pedido. Ele fica só na memória desta sessão. Apenas `train.csv` e `train_images/` são extraídos, pois `test_images/` não tem rótulos públicos.

In [ ]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Token da API do Kaggle: ")

!pip install -q kaggle
!kaggle competitions download -c aptos2019-blindness-detection -p data/aptos
!unzip -q -o data/aptos/aptos2019-blindness-detection.zip train.csv "train_images/*" -d data/aptos
!rm data/aptos/aptos2019-blindness-detection.zip
!ls data/aptos/train_images | wc -l

## 4. Teste rápido

Uma época com 100 imagens, para achar problemas antes do treino longo. O resultado não serve como métrica.

In [ ]:
!python -m src.dr.train --data data/aptos --epochs 1 --limit 100 --workers 2 --out models/smoke.pt

## 5. Treino completo

In [ ]:
!python -m src.dr.train --data data/aptos --epochs 15 --workers 2 --out models/best.pt

## 6. Avaliação e exemplos de Grad-CAM

Grava `docs/metrics.json`, `docs/confusion_matrix.png` e `docs/gradcam_grau*.png`.

In [ ]:
!python -m src.dr.evaluate --data data/aptos --weights models/best.pt --workers 2
!python scripts/gradcam_demo.py --data data/aptos --weights models/best.pt

## 7. Baixa os resultados

Coloque `best.pt` em `models/` e o conteúdo de `docs/` em `docs/` no seu repositório.

In [ ]:
!zip -q -r resultados.zip models/best.pt docs

from google.colab import files

files.download("resultados.zip")

### Opcional: copia para o Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/dr-screening
!cp resultados.zip /content/drive/MyDrive/dr-screening/